# 01 - Clasificación tabular profunda en Airlines

Este notebook compara, con un protocolo experimental común, un baseline lineal y cuatro arquitecturas de deep learning tabular: TabNetClassifier, TabTransformer, FT-Transformer y SAINT supervisado. El propósito es observar cómo representan datos mixtos, cómo convergen y qué costo computacional requieren.

La implementación prioriza reproducibilidad, auditabilidad, ausencia de fuga de información y una lectura prudente de los resultados. No se realiza una búsqueda exhaustiva de hiperparámetros ni se incorporan técnicas ajenas a la comparación arquitectónica.


## 1. Título, contexto y objetivos

Trabajamos con el dataset Airlines Passenger Satisfaction. El objetivo supervisado es predecir `satisfaction`; en el caso binario la clase positiva se define como `satisfied`, es decir, un pasajero que reporta satisfacción con la experiencia de vuelo.

El notebook cumple cinco funciones: explicar el razonamiento experimental, centralizar la configuración, ejecutar los modelos, visualizar resultados y discutir limitaciones sin afirmar superioridad antes de observar evidencia.


## 2. Preguntas experimentales

1. ¿Los modelos tabulares profundos superan a un baseline lineal sencillo bajo los mismos splits y métricas?
2. ¿Qué arquitectura ofrece mejor balance entre rendimiento, costo computacional y estabilidad?
3. ¿La accuracy cuenta una historia suficiente cuando las clases no están perfectamente balanceadas?
4. ¿Cómo difieren las representaciones internas producidas por cada arquitectura?
5. ¿Qué decisiones metodológicas son necesarias para comparar varias semillas sin contaminar test?


## 3. Importaciones

El notebook importa solo código de presentación y funciones de alto nivel. La lógica de datos, modelos, entrenamiento y evaluación vive en `src/`, lo cual reduce el riesgo de ejecutar celdas fuera de orden o duplicar pipelines por arquitectura.


In [ ]:
from importlib.metadata import version
from pathlib import Path
import logging

import pandas as pd
from IPython.display import display

from src.data import (
    SplitConfig,
    audit_dataframe,
    load_classification_dataframe,
    make_airlines_config,
    prepare_classification_data,
    sample_classification_dataframe,
)
from src.evaluation import (
    comparison_table,
    plot_confusion_matrices,
    plot_roc_pr_curves,
    validate_experiment_results,
)
from src.models import optional_dependency_report
from src.training import (
    finalize_classification_experiment,
    resolve_device,
    run_classification_experiment,
    set_reproducible_seed,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
pd.set_option("display.max_columns", 120)

DEPENDENCY_REPORT = optional_dependency_report()
missing_dependencies = [name for name, available in DEPENDENCY_REPORT.items() if not available]
if missing_dependencies:
    raise ImportError(f"Missing required dependencies: {missing_dependencies}")
DEPENDENCY_VERSIONS = {
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "scikit-learn": version("scikit-learn"),
    "torch": version("torch"),
    "pytorch-tabnet": version("pytorch-tabnet"),
}
DEPENDENCY_VERSIONS


## 4. Configuración central

Todos los valores que afectan reproducibilidad, rutas, presupuesto de entrenamiento y configuración base de modelos se concentran aquí. No hay hiperparámetros dispersos en las secciones posteriores.

Se usan tres semillas de entrenamiento sobre un único split fijo. Así se estima variabilidad debida a inicialización y minibatches sin cambiar las filas asignadas a train, validación y test.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the repository root."
    )

DATA_DIR = PROJECT_ROOT / "archive"
RESULTS_DIR = PROJECT_ROOT / "results" / "classification"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

EXPERIMENT_CONFIG = {
    "experiment_id": "airlines_dl_architectures_v2",
    "seeds": [42, 123, 2025],
    "seed": 42,
    "device": "auto",
    "deterministic": True,
    "sample_size": None,
    "batch_size": 1024,
    "inference_batch_size": 4096,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "max_epochs": 40,
    "patience": 8,
    "num_workers": 0,
    "selection_metric": "roc_auc",
    "threshold": 0.5,
    "optimize_threshold": False,
    "threshold_metric": "f1",
    "defer_test_until_final": True,
    "require_row_independent_inference": True,
    "dependency_versions": DEPENDENCY_VERSIONS,
    "results_dir": RESULTS_DIR,
    "model_configs": {
        "baseline_logistic": {"max_iter": 1000, "solver": "lbfgs"},
        "tabnet": {
            "n_d": 24,
            "n_a": 24,
            "n_steps": 4,
            "gamma": 1.4,
            "lambda_sparse": 1e-4,
            "cat_emb_dim": 8,
        },
        "tab_transformer": {
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
            "mlp_hidden": (128, 64),
        },
        "ft_transformer": {
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
        },
        "saint_supervised": {
            "implementation_version": "saint_column_v1",
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
            "mlp_hidden": (128, 64),
            "numerical_embedding_hidden": 32,
            "use_row_attention": False,
        },
    },
}

DATASET_CONFIG = make_airlines_config(DATA_DIR)
SPLIT_CONFIG = SplitConfig(train_size=0.70, valid_size=0.15, test_size=0.15, random_state=EXPERIMENT_CONFIG["seed"])
MODEL_ORDER = ["baseline_logistic", "tabnet", "tab_transformer", "ft_transformer", "saint_supervised"]


## 5. Semillas y dispositivo

La reproducibilidad en deep learning depende de inicialización, orden de minibatches y kernels numéricos. Se fijan semillas para Python, NumPy, PyTorch y CUDA. En modo determinista se desactivan los kernels Flash y memory-efficient de atención, y una operación no determinista restante produce un error explícito. Distintas versiones o dispositivos aún pueden introducir pequeñas diferencias numéricas, por lo que se comparan varias semillas.


In [ ]:
set_reproducible_seed(EXPERIMENT_CONFIG["seed"], deterministic=EXPERIMENT_CONFIG["deterministic"])
DEVICE = resolve_device(EXPERIMENT_CONFIG["device"])
EXPERIMENT_CONFIG["device"] = DEVICE

print(f"Device resolved for this run: {DEVICE}")
print("Optional dependency report:", optional_dependency_report())


## 6. Carga del dataset

Se cargan los archivos `train.csv` y `test.csv` disponibles en `archive/`, pero para esta comparación se concatenan y se vuelve a construir una partición 70/15/15 estratificada. Esta decisión hace explícito que todos los modelos usan exactamente los mismos splits controlados por la semilla del experimento.


In [ ]:
raw_df = load_classification_dataframe(DATASET_CONFIG)
print(f"Dataset shape: {raw_df.shape}")
display(raw_df.head())


## 7. Auditoría estructural y de calidad

Antes de modelar, verificamos dimensiones, tipos, valores faltantes, distribución de clases e identificadores. Esta auditoría no ajusta transformadores ni mira el conjunto de prueba como fuente de decisiones de entrenamiento.


In [ ]:
audit = audit_dataframe(raw_df, DATASET_CONFIG)
print("Shape:", audit["shape"])
print("Missing values:", audit["missing_values"])
print("Identifier uniqueness:", audit["identifier_uniqueness"])
print("Configured leakage-sensitive columns:", audit["configured_leakage_columns"])
display(pd.DataFrame.from_dict(audit["target_distribution"], orient="index", columns=["count"]))


## 8. Definición de target y características

El target es `satisfaction`; la clase positiva es `satisfied`. Se excluyen `Unnamed: 0` e `id` porque son identificadores o artefactos de exportación, no variables explicativas.

Las variables categóricas nominales se codifican con un índice reservado para categorías desconocidas. Las variables de rating 0-5 se tratan inicialmente como numéricas/ordinales, una elección razonable para esta base clásica porque preserva su orden natural. La configuración marca ratings y retrasos como `leakage-sensitive`: se conservan para modelar satisfacción retrospectiva, pero deberían excluirse si el horizonte fuese previo a la experiencia de vuelo.


In [ ]:
print("Target:", DATASET_CONFIG.target_name)
print("Positive class:", DATASET_CONFIG.positive_class)
print("Positive class meaning:", DATASET_CONFIG.positive_class_meaning)
print("Excluded columns:", DATASET_CONFIG.excluded_columns)
print("Categorical columns:", DATASET_CONFIG.categorical_columns)
print("Numerical columns:", DATASET_CONFIG.numerical_columns)


## 9. Análisis breve de la distribución de clases

Accuracy puede ser insuficiente si un modelo favorece la clase mayoritaria. Por eso además se reportan balanced accuracy, precision, recall, F1, ROC-AUC y PR-AUC. En problemas binarios, ROC-AUC y PR-AUC deben calcularse con probabilidades, no con etiquetas discretas; aquí PR-AUC se resume mediante average precision.


In [ ]:
class_distribution = raw_df[DATASET_CONFIG.target_name].value_counts().rename("count").to_frame()
class_distribution["percent"] = 100 * class_distribution["count"] / class_distribution["count"].sum()
display(class_distribution)


## 10. Partición train/validation/test

La validación y la prueba se separan porque cumplen roles distintos. Validación se usa para early stopping, selección del mejor checkpoint y ajuste opcional del umbral. Prueba queda aislada hasta la evaluación final para estimar generalización sin reutilizar evidencia.

Esta exploración usa división aleatoria estratificada 70/15/15. El objeto `SplitConfig` mantiene separada la estrategia de partición para que otros datasets puedan requerir grupos o bloques temporales sin alterar la lógica de modelos.


In [ ]:
working_df = sample_classification_dataframe(
    raw_df,
    target_name=DATASET_CONFIG.target_name,
    sample_size=EXPERIMENT_CONFIG["sample_size"],
    random_state=EXPERIMENT_CONFIG["seed"],
)

prepared_data = prepare_classification_data(working_df, DATASET_CONFIG, SPLIT_CONFIG)
print(prepared_data.split_report["split_sizes"])
display(pd.DataFrame(prepared_data.split_report["class_distribution"]).T)


## 11. Ajuste del preprocesamiento únicamente con train

La imputación, la normalización y los mapas categóricos se ajustan solo con entrenamiento. Si validación o prueba influyeran en medianas, medias, desviaciones estándar o vocabularios, el modelo recibiría información indirecta de datos que debe simular no haber visto.

Las categorías desconocidas se codifican como índice 0. Esto evita fallos en inferencia y separa explícitamente lo no observado en entrenamiento de las categorías conocidas.


In [ ]:
print("Feature matrix shapes:")
print("  train:", prepared_data.X_train.shape)
print("  valid:", prepared_data.X_valid.shape)
print("  test :", prepared_data.X_test.shape)
print("Categorical cardinalities:", prepared_data.categorical_cardinalities)
print("Preprocessing fitted on:", prepared_data.preprocessing_state.fitted_on)
print("Fit row count:", prepared_data.preprocessing_state.fit_row_count)


## 12. Verificaciones contra leakage

Las siguientes verificaciones son aserciones programáticas, no mensajes decorativos: no hay índices compartidos, no quedan NaN tras preprocesar, las etiquetas son válidas, las cardinalidades respetan el vocabulario de entrenamiento y el split de prueba no participa en selección de modelos.


In [ ]:
checks = prepared_data.split_report
reprepared_data = prepare_classification_data(working_df, DATASET_CONFIG, SPLIT_CONFIG)

assert checks["no_index_overlap"]
assert checks["all_rows_assigned_once"]
assert checks["no_nan_after_preprocessing"]
assert checks["labels_are_valid"]
assert checks["cardinalities_are_valid"]
assert checks["preprocessing_fitted_only_on_train"]
assert checks["test_split_used_for_model_selection"] is False
assert checks["class_proportions_preserved"]
assert prepared_data.train_indices.tolist() == reprepared_data.train_indices.tolist()
assert prepared_data.valid_indices.tolist() == reprepared_data.valid_indices.tolist()
assert prepared_data.test_indices.tolist() == reprepared_data.test_indices.tolist()
assert prepared_data.split_fingerprint() == reprepared_data.split_fingerprint()

checks["same_seed_split_reproducibility"] = True
checks


## 13. Baseline clásico

El baseline es una regresión logística con one-hot encoding de categorías ya codificadas y variables numéricas normalizadas. Su inductive bias es lineal: aprende fronteras simples y sirve como referencia honesta para preguntar si las arquitecturas profundas aportan valor real.

No es un modelo profundo, pero permite reconocer cuánto de la señal puede capturarse mediante relaciones lineales antes de interpretar las arquitecturas de mayor capacidad.


In [ ]:
experiment_results = []
results_by_model = {}


def validation_table(model_results: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["model_name"],
                "seed": result["seed"],
                "best_epoch": result["best_epoch"],
                **result["valid_metrics"],
            }
            for result in model_results
        ]
    )


def execute_model(model_name: str) -> pd.DataFrame:
    model_results = []
    for seed in EXPERIMENT_CONFIG["seeds"]:
        run_config = dict(EXPERIMENT_CONFIG)
        run_config["seed"] = seed
        model_result = run_classification_experiment(
            model_name=model_name,
            data=prepared_data,
            config=run_config,
        )
        model_results.append(model_result)
    results_by_model[model_name] = model_results
    experiment_results.extend(model_results)
    return validation_table(model_results)

baseline_table = execute_model("baseline_logistic")
display(baseline_table)


## 14. TabNetClassifier

TabNet selecciona características mediante máscaras atencionales secuenciales. Su sesgo inductivo favorece decisiones dispersas por pasos, parecidas a una selección suave de variables. En este pipeline recibe categorías codificadas con cardinalidades conocidas por entrenamiento y variables numéricas normalizadas.

Ventajas esperadas: interpretabilidad por máscaras y buen comportamiento en datos tabulares mixtos. Limitaciones: entrenamiento sensible a hiperparámetros y mayor costo que el baseline. Este wrapper conserva la interfaz común, pero no presenta las máscaras como si fueran una representación latente única y estable.


In [ ]:
tabnet_table = execute_model("tabnet")
display(tabnet_table)


## 15. TabTransformer

TabTransformer contextualiza principalmente las categorías: cada columna categórica se embebe como token y un Transformer aprende interacciones entre esas categorías. Luego se concatena esa representación contextual con las variables numéricas normalizadas antes de la cabeza clasificadora.

Su inductive bias es que las relaciones entre categorías importan y no deben tratarse como columnas independientes. Puede fallar si la señal principal está en variables continuas complejas, porque los numéricos no se tokenizan con la misma simetría que en FT-Transformer. La salida previa a la cabeza permite inspeccionar la representación contextual aprendida.


In [ ]:
tab_transformer_table = execute_model("tab_transformer")
display(tab_transformer_table)


## 16. FT-Transformer

FT-Transformer tokeniza tanto variables categóricas como numéricas. Cada feature se convierte en un token y un token `[CLS]` agrega la información para clasificación. A diferencia de TabTransformer, las columnas continuas participan desde el inicio en la atención.

Su ventaja esperada es modelar interacciones columna-columna de forma más uniforme. Su limitación principal es el costo de atención y la necesidad de regularización si el dataset no exige tanta capacidad. El token `[CLS]` ofrece una representación compacta previa a la cabeza de clasificación.


In [ ]:
ft_transformer_table = execute_model("ft_transformer")
display(ft_transformer_table)


## 17. SAINT supervisado

SAINT representa cada columna como token y su diseño completo puede alternar atención entre columnas y entre filas. La atención intersample aporta contexto entre observaciones, pero también hace que una predicción dependa de qué otras filas comparten el batch.

La comparación principal usa la variante supervisada inductiva `saint_column_v1`, con atención entre columnas y `use_row_attention=False`. De este modo cada salida depende solo de su propia fila, como en TabNet, TabTransformer y FT-Transformer. La implementación intersample permanece disponible como variante explícita, pero no se mezcla en este ranking porque responde a un protocolo transductivo diferente.


In [ ]:
saint_table = execute_model("saint_supervised")
display(saint_table)


## 18. Evaluación final sobre test

Hasta este punto cada llamada a `run_classification_experiment` utilizó únicamente train y validación. Los hiperparámetros, umbral y checkpoints ya están fijados. En esta sección se restauran todos los checkpoints y se abre test una sola vez; después, una auditoría recompone métricas y verifica probabilidades, etiquetas, índices y huella del split.


In [ ]:
experiment_results = [
    finalize_classification_experiment(result, prepared_data)
    for result in experiment_results
]
integrity_report = validate_experiment_results(experiment_results)
display(pd.Series(integrity_report, name="integrity_check"))

all_results = comparison_table(experiment_results)
display(all_results.sort_values("roc_auc", ascending=False))


## 19. Tabla comparativa

La tabla resume media y desviación estándar sobre tres semillas de entrenamiento. Todas usan el mismo split; por tanto, la dispersión refleja inicialización y orden de minibatches, no cambios en la composición de test.


In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "train_time_seconds",
    "inference_time_seconds",
    "n_parameters",
]
summary = all_results.groupby(
    ["model_name", "implementation_version"]
)[metric_columns].agg(["mean", "std"])
display(summary)


## 20. Curvas ROC y precision-recall

ROC resume sensibilidad frente a falsos positivos usando probabilidades. Precision-recall es especialmente informativa cuando interesa el rendimiento sobre la clase positiva y cuando las proporciones no son perfectamente simétricas. Para mantener legibilidad, las curvas muestran la primera semilla declarada; la tabla anterior contiene el resumen de las tres.


In [ ]:
PLOT_SEED = EXPERIMENT_CONFIG["seeds"][0]
plot_results = [
    result for result in experiment_results if result["seed"] == PLOT_SEED
]
curve_paths = plot_roc_pr_curves(plot_results, FIGURES_DIR)
curve_paths


## 21. Matrices de confusión

La matriz de confusión muestra qué tipo de error comete cada modelo: falsos positivos de satisfacción o falsos negativos de satisfacción. Se utiliza la misma semilla representativa de las curvas; la variabilidad global se consulta en la tabla agregada.


In [ ]:
confusion_path = plot_confusion_matrices(plot_results, FIGURES_DIR)
confusion_path


## 22. Comparación de costo computacional

El costo no es secundario: un modelo más preciso pero mucho más lento o grande puede no ser conveniente para experimentos repetidos o barridos de semillas. Se comparan parámetros entrenables, tiempo de entrenamiento e inferencia.


In [ ]:
cost_table = all_results[[
    "model_name",
    "implementation_version",
    "seed",
    "n_parameters",
    "train_time_seconds",
    "inference_time_seconds",
    "best_epoch",
    "epochs_trained",
    "reached_epoch_budget",
]]
display(cost_table.sort_values("train_time_seconds"))


## 23. Discusión teórica

La comparación no fuerza a todas las arquitecturas a recibir la misma representación interna, porque hacerlo borraría parte de su diseño. TabNet usa embeddings para categorías y máscaras de selección; TabTransformer contextualiza categorías; FT-Transformer tokeniza todas las columnas; la variante inductiva de SAINT aplica embeddings no lineales y atención entre columnas.

Lo justo no es homogeneizar la arquitectura, sino mantener constantes los splits, etiquetas, semillas, métrica de selección, presupuesto general y aislamiento del test. Los modelos PyTorch usan el mismo tamaño de batch; TabNet conserva ghost batch normalization mediante `virtual_batch_size`, porque forma parte de su diseño.


## 24. Limitaciones

Esta ejecución usa configuración base, no búsqueda exhaustiva de hiperparámetros. El rendimiento observado no debe interpretarse como ranking definitivo. Además, Airlines contiene variables post-experiencia como ratings de servicio; son válidas si el problema se formula como modelado de satisfacción observada, pero no si se pretendiera predecir satisfacción antes del vuelo.

Tres semillas permiten estimar una variabilidad inicial, aunque siguen siendo insuficientes para una afirmación definitiva. Los valores 0 de las escalas de servicio se conservan como ordinales; antes de una publicación debe confirmarse en el diccionario original si significan puntuación mínima o respuesta no aplicable. La configuración base tampoco sustituye una búsqueda controlada de hiperparámetros.


## 25. Extensión a otros datasets

La estrategia de partición debe responder a la estructura de cada dataset. Si existen sujetos, sesiones, archivos o bloques temporales correlacionados, una división aleatoria puede mezclar observaciones casi idénticas entre train y test.

Por eso `SplitConfig` separa la estrategia de partición de la lógica de modelos. Una estrategia por grupos o por tiempo puede incorporarse sin reescribir wrappers, entrenamiento ni evaluación.


## 26. Representaciones internas

TabTransformer, FT-Transformer y SAINT exponen `get_embedding` mediante sus wrappers PyTorch. Estas representaciones se obtienen antes de la cabeza clasificadora y permiten estudiar separabilidad, estabilidad y dimensionalidad sin confundirlas con probabilidades finales.

La extracción debe realizarse con el checkpoint elegido por validación y con el modelo en modo de evaluación. En la variante inductiva de SAINT, el embedding de una fila no cambia al reorganizar el resto del conjunto.


## 27. Conclusiones

El notebook deja una comparación clásica reproducible y extensible: datos preparados sin leakage, modelos con interfaz común, early stopping por validación, checkpoints restaurables, métricas probabilísticas y artefactos persistidos automáticamente.

Las conclusiones sustantivas deben extraerse de la tabla y curvas producidas por la ejecución real. Si se amplía a varias semillas, el criterio principal debe considerar rendimiento promedio, variabilidad y costo computacional, no solo la mejor métrica puntual.
